# 01. 데이터 전처리

원본 데이터 5종의 중복·결측·시간대를 정리하고 사용자 단위 기본 분석 테이블을 만듭니다.

> **주의:** 원본 CSV는 GitHub에 포함하지 않습니다. 로컬 `data/raw/`에 직접 배치하세요.

**입력:** `data/raw/*.csv`  
**출력:** `data/interim/*.csv`, `data/processed/master_base.csv`

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    if current.name == "notebooks":
        return current.parent
    if (current / "notebooks").exists():
        return current
    return current.parent

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SAMPLE_DIR = PROJECT_ROOT / "data" / "sample"
MODEL_DIR = PROJECT_ROOT / "models"

for directory in [INTERIM_DIR, PROCESSED_DIR, SAMPLE_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd

raw_files = {
    "site": RAW_DIR / "site_area.csv",
    "register": RAW_DIR / "trial_register.csv",
    "visit": RAW_DIR / "trial_visit_info.csv",
    "log": RAW_DIR / "trial_access_log.csv",
    "payment": RAW_DIR / "trial_payment.csv",
}

missing = [str(path) for path in raw_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "원본 데이터는 GitHub에 포함되지 않습니다. "
        "아래 파일을 로컬 data/raw/에 넣어주세요:\n- " + "\n- ".join(missing)
    )

site = pd.read_csv(raw_files["site"])
register = pd.read_csv(raw_files["register"])
visit = pd.read_csv(raw_files["visit"])
log = pd.read_csv(raw_files["log"])
payment = pd.read_csv(raw_files["payment"])

pd.DataFrame({
    "table": ["site_area", "trial_register", "trial_visit_info", "trial_access_log", "trial_payment"],
    "rows": [len(site), len(register), len(visit), len(log), len(payment)],
    "cols": [site.shape[1], register.shape[1], visit.shape[1], log.shape[1], payment.shape[1]],
})

## 정제 기준

| 데이터 | 처리 |
|---|---|
| register | 중복 제거 후 사용자별 마지막 신청 기준 |
| visit_info | 입·퇴실 시각 동시 결측 제거 + 완전 중복 제거 |
| access_log | 완전 중복 제거 + UTC 기준 시각을 KST(+9h)로 변환 |
| payment | 사용자별 중복 제거 |

In [ ]:
# register
register_clean = register.copy()
register_clean["trial_date"] = pd.to_datetime(register_clean["trial_date"], errors="coerce")
register_clean = (
    register_clean
    .drop_duplicates()
    .sort_values(["user_uuid", "trial_date"])
    .drop_duplicates("user_uuid", keep="last")
    .reset_index(drop=True)
)

# payment
payment_clean = (
    payment
    .drop_duplicates()
    .drop_duplicates("user_uuid", keep="last")
    .reset_index(drop=True)
)

# visit
visit_clean = visit.copy()
both_missing = (
    visit_clean["first_enter_time"].isna()
    & visit_clean["last_leave_time"].isna()
)
visit_clean = visit_clean.loc[~both_missing].drop_duplicates().copy()
visit_clean["date"] = pd.to_datetime(visit_clean["date"], errors="coerce")

# access log
log_clean = log.drop_duplicates().copy()
log_clean["cdate"] = pd.to_datetime(log_clean["cdate"], errors="coerce")
log_clean["cdate_kst"] = log_clean["cdate"] + pd.Timedelta(hours=9)

summary = pd.DataFrame({
    "table": ["register", "visit_info", "access_log", "payment"],
    "raw_rows": [len(register), len(visit), len(log), len(payment)],
    "clean_rows": [len(register_clean), len(visit_clean), len(log_clean), len(payment_clean)],
})
summary

In [ ]:
# 사용자 단위 방문 요약
visit_agg = (
    visit_clean
    .groupby("user_uuid")
    .agg(
        visit_days=("date", "nunique"),
        total_stay_time=("stay_time_second", "sum"),
        first_visit_date=("date", "min"),
    )
    .reset_index()
)

# 가장 많이 방문한 지점을 주 이용 지점으로 정의
site_count = (
    visit_clean
    .groupby(["user_uuid", "site_id"])
    .size()
    .reset_index(name="visit_count")
    .sort_values(
        ["user_uuid", "visit_count", "site_id"],
        ascending=[True, False, True],
    )
)

primary_site = (
    site_count
    .drop_duplicates("user_uuid")
    .merge(site, on="site_id", how="left")
    [["user_uuid", "site_id", "area_pyeong"]]
    .rename(columns={"site_id": "primary_site"})
)

master_base = (
    register_clean
    .merge(payment_clean, on="user_uuid", how="inner")
    .merge(visit_agg, on="user_uuid", how="inner")
    .merge(primary_site, on="user_uuid", how="left")
)

master_base["first_visit_delay"] = (
    master_base["first_visit_date"] - master_base["trial_date"]
).dt.days

print("전체 신청자:", len(register_clean))
print("방문 기록 보유 사용자:", len(master_base))
print("방문일수 분포:")
print(master_base["visit_days"].value_counts().sort_index())

In [ ]:
# 개인정보성 식별키가 포함된 중간 산출물도 GitHub에는 올리지 않는 것을 권장합니다.
register_clean.to_csv(INTERIM_DIR / "register_clean.csv", index=False, encoding="utf-8-sig")
visit_clean.to_csv(INTERIM_DIR / "visit_clean.csv", index=False, encoding="utf-8-sig")
log_clean.to_csv(INTERIM_DIR / "access_log_clean.csv", index=False, encoding="utf-8-sig")
payment_clean.to_csv(INTERIM_DIR / "payment_clean.csv", index=False, encoding="utf-8-sig")
master_base.to_csv(PROCESSED_DIR / "master_base.csv", index=False, encoding="utf-8-sig")

print("저장 완료")